# 02 Cleaning

This notebook standardizes the extracted dataset, applies the cleaning rules for numeric conversion and missingness handling, validates OHLC consistency, engineers derived trading features, and writes the cleaned outputs plus the ETL quality report.

In [1]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from scripts.etl_pipeline import (
    add_features,
    clean_numeric_fields,
    generate_quality_report,
    load_raw_stock_files,
    save_outputs,
    standardize_columns,
)

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs" / "tables"
DOCS_DIR = REPO_ROOT / "docs"


## Load Extracted Data

The cleaning notebook expects the extracted master file from Notebook 01. If the file is missing, the cell raises a clear instruction instead of silently fabricating a downstream dataset.

In [2]:
raw_combined_path = PROCESSED_DIR / "nifty50_combined_raw.csv"
if not raw_combined_path.exists():
    raise FileNotFoundError(
        "Run notebooks/01_extraction.ipynb or python scripts/etl_pipeline.py first to create "
        "data/processed/nifty50_combined_raw.csv."
    )

raw_combined = pd.read_csv(raw_combined_path)
raw_combined.shape

(235192, 21)

## Standardize and Clean

This step converts the schema to `snake_case`, parses the trading date, converts required numeric fields safely, removes exact duplicates, applies symbol-wise forward/backward filling only to price fields, keeps missingness in trading-activity columns visible, and flags any OHLC inconsistencies.

In [3]:
standardized = standardize_columns(raw_combined)
cleaned, cleaning_audit = clean_numeric_fields(standardized)

print("Standardized columns:")
print(list(cleaned.columns))
print("\nMissing before cleaning:")
display(cleaning_audit["missing_before"][cleaning_audit["missing_before"] > 0].to_frame("missing_rows"))
print("\nMissing after cleaning:")
display(cleaning_audit["missing_after"][cleaning_audit["missing_after"] > 0].to_frame("missing_rows"))
print("\nTurnover scale factor applied:", cleaning_audit["turnover_scale_factor"])
print("Exact duplicates removed:", cleaning_audit["exact_duplicates_removed"])
print("Invalid OHLC rows:", cleaning_audit["invalid_ohlc_rows"])


Standardized columns:
['date', 'symbol', 'series', 'prev_close', 'open', 'high', 'low', 'last', 'close', 'vwap', 'volume', 'turnover', 'trades', 'deliverable_volume', 'deliverable_percent', 'source_file', 'source_symbol', 'company_name', 'industry', 'isin_code', 'metadata_join_key', 'missing_price_flag', 'missing_volume_flag', 'invalid_ohlc_flag']

Missing before cleaning:


,missing_rows
trades,114848
deliverable_percent,16077
deliverable_volume,16077



Missing after cleaning:


,missing_rows
trades,114848
deliverable_percent,16077
deliverable_volume,16077



Turnover scale factor applied: 100000.0
Exact duplicates removed: 0
Invalid OHLC rows: 0


## Engineer Features and Flags

The derived fields below support later risk, liquidity, and sector analysis while keeping quality flags visible for downstream filtering and auditability.

In [4]:
cleaned_features, feature_audit = add_features(cleaned)

flag_summary = pd.Series(
    {
        "missing_price_flag": int(cleaned_features["missing_price_flag"].sum()),
        "missing_volume_flag": int(cleaned_features["missing_volume_flag"].sum()),
        "invalid_ohlc_flag": int(cleaned_features["invalid_ohlc_flag"].sum()),
        "outlier_return_flag": int(cleaned_features["outlier_return_flag"].sum()),
    },
    name="flagged_rows",
)

display(flag_summary.to_frame())
cleaned_features[[
    "symbol",
    "date",
    "daily_return",
    "intraday_return",
    "high_low_spread",
    "vwap_gap",
    "turnover_cr",
    "delivery_ratio",
    "outlier_return_flag",
]].head()

,flagged_rows
missing_price_flag,0
missing_volume_flag,114848
invalid_ohlc_flag,0
outlier_return_flag,12680


,symbol,date,daily_return,intraday_return,high_low_spread,vwap_gap,turnover_cr,delivery_ratio,outlier_return_flag
0,ADANIPORTS,2012-01-17,0.033210,0.021152,6.00,1.87,22.600735,0.6138,False
1,ADANIPORTS,2012-01-18,0.012143,-0.002113,5.10,0.45,12.579861,0.4547,False
2,ADANIPORTS,2012-01-19,0.054340,0.037500,7.40,2.68,21.363823,0.4955,True
3,ADANIPORTS,2012-01-20,0.040161,0.023041,7.35,1.64,25.125832,0.5270,False
4,ADANIPORTS,2012-01-23,-0.055663,-0.055663,10.30,-2.79,24.787678,0.4951,True


## Save Cleaned Outputs

The final cell regenerates the extraction report for documentation context, writes the cleaned CSV, summary table, and Markdown cleaning log, and prints the saved output paths.

In [5]:
combined_raw_for_report, _, extraction_report = load_raw_stock_files(RAW_DIR)
cleaning_summary, quality_report = generate_quality_report(
    combined_raw=combined_raw_for_report,
    cleaned=cleaned_features,
    extraction_report=extraction_report,
    cleaning_audit=cleaning_audit,
    feature_audit=feature_audit,
)

output_paths = save_outputs(
    combined_raw=combined_raw_for_report,
    cleaned=cleaned_features,
    cleaning_summary=cleaning_summary,
    quality_report=quality_report,
    processed_dir=PROCESSED_DIR,
    outputs_dir=OUTPUTS_DIR,
    docs_dir=DOCS_DIR,
)

for label, path in output_paths.items():
    print(f"{label}: {path}")

cleaning_summary.head(15)

raw_output: /Users/devaanshkathuria/Documents/Projects/DVA_Final/DVA_capstone_2/data/processed/nifty50_combined_raw.csv
cleaned_output: /Users/devaanshkathuria/Documents/Projects/DVA_Final/DVA_capstone_2/data/processed/nifty50_cleaned.csv
summary_output: /Users/devaanshkathuria/Documents/Projects/DVA_Final/DVA_capstone_2/outputs/tables/cleaning_summary.csv
report_output: /Users/devaanshkathuria/Documents/Projects/DVA_Final/DVA_capstone_2/docs/cleaning_log.md


,metric,value
0,raw_rows_combined,235192
1,cleaned_rows,235192
2,stock_files_found,50
3,combined_stock_files_found,1
4,metadata_available,True
5,rows_with_metadata,235192
6,rows_without_metadata,0
7,exact_duplicates_removed,0
8,rows_with_price_missing_before_fill,0
9,price_missing_cells_filled,0
